In [3]:
from glob import glob
import pandas as pd
from os import path
import seaborn as sns
import numpy as np
np.set_printoptions(legacy='1.21')
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scores = defaultdict(dict)
for f in glob("benchmark_exp/score/uni/*/*.npy"):
    #print(f)
    new_scores = np.load(f)
    algorithm = f.split("/")[3]
    data_file = path.basename(f).split(".")[0]
    scores[data_file][algorithm] = new_scores

In [ ]:
scores['747_SMD_processed_test_machine-2-8_5925_17580_col_id_12_5925_17580'].keys()

In [6]:
portfolio = ['median_dist_score', 'IForest', 'Sub_KMeansAD', 'Sub_MCD', 'POLY', 'median_dist_rolling_score', 'Sub_PCA', 'diff_diff_std_score']
portfolio_big = ['Sub_PCA_projection2', 'Sub_KNN', 'SR_with_window_size', 'median_dist_rolling_score2', 'median_dist_score', 'IForest', 'kpi_custom_average', 'SR_better_window_size', 'mean_dist_score', 'Sub_KMeansAD', 'diff_rolling_score', 'Sub_PCA_projection', 'raw_signal', 'Sub_HBOS', 'Decompose', 'Sub_MCD', 'POLY', 'KNN', 'Sub_LOF', 'rolling_3', 'SR', 'Sub_IForest', 'median_dist_rolling_score', 'MatrixProfile', 'Sub_PCA_projection_quo_vadis', 'Sub_PCA_projection_quo_vadis2', 'Sub_PCA', 'diff_diff_std_score', 'negative_raw_signal', 'MatrixProfile1NoNormalize', 'PCA', 'SAND', 'rolling_window_score', 'rolling_diff_mult_1_5', 'AutoEncoder', 'HBOS', 'ucr_custom_average', 'negative_rolling_window_score']

In [15]:
portfolio_scores = {}
portfolio_scores_std = {}
portfolio_scores_no_scale = {}

In [16]:
from sklearn.preprocessing import MinMaxScaler

for file, these_scores in scores.items():
    scores_array = np.concatenate([x.reshape(-1, 1) for n, x in these_scores.items() if n in portfolio], axis=1)
    portfolio_scores[file] = MinMaxScaler().fit_transform(scores_array).mean(axis=1)
    portfolio_scores_std[file] = StandardScaler().fit_transform(scores_array).mean(axis=1)
    portfolio_scores_no_scale[file] = scores_array.mean(axis=1)


In [8]:
eval_files = pd.read_csv("~/public/TSB-AD/benchmark_exp/TSB-AD/File-List/TSB-AD-U-Eval-List.csv")

files = {}
for f in eval_files.file_name:
    files[f.split(".")[0]] = pd.read_csv(f"~/public/TSB-AD/benchmark_exp/TSB-AD/TSB-AD-U/{f}")

In [8]:
from sklearn.metrics import average_precision_score
from tqdm import tqdm
import os

ap_scores = {}
for file, df  in tqdm(files.items()):
    ad_scores = portfolio_scores[file]
    ap_scores[file] = average_precision_score(df.Label, ad_scores)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 310/310 [00:12<00:00, 24.28it/s]


In [20]:

ap_scores_no_scale = {}
for file, df  in tqdm(files.items()):
    ad_scores = portfolio_scores_no_scale[file]
    ap_scores_no_scale[file] = average_precision_score(df.Label, ad_scores)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 310/310 [00:08<00:00, 37.88it/s]


In [21]:
ap_scores_std = {}
for file, df  in tqdm(files.items()):
    ad_scores_std = portfolio_scores_std[file]
    ap_scores_std[file] = average_precision_score(df.Label, ad_scores_std)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 310/310 [00:09<00:00, 34.13it/s]


In [9]:
df = pd.Series(ap_scores)
df

001_NAB_data_Traffic_4_624_2087           0.858274
003_NAB_data_Exchange_4_500_1045          0.226572
004_NAB_data_art1_0_1007_2679             0.724208
005_NAB_data_Traffic_7_623_2084           0.612689
006_NAB_data_CloudWatch_11_1007_1437      0.431140
                                            ...   
825_ECG_MBA_ECG14046_data_23_2112_2212    0.435338
826_ECG_MBA_ECG14046_data_30_8869_8969    0.617247
827_ECG_MBA_ECG14046_data_27_726_826      0.384929
828_ECG_MBA_ECG14046_data_15_1832_1932    0.484971
829_ECG_MBA_ECG14046_data_44_2508_2608    0.218683
Length: 310, dtype: float64

In [10]:
df.to_csv("pr_ensemble_test.csv")

In [11]:
df.mean()

0.4430130812415

In [12]:
df.std()

0.3401707913456048

In [27]:

pd.DataFrame([ap_scores, ap_scores_no_scale, ap_scores_std]).rank().T.mean()

0    2.072581
1    1.648387
2    2.279032
dtype: float64

In [28]:
pd.DataFrame([ap_scores, ap_scores_no_scale, ap_scores_std]).T.mean()

0    0.443013
1    0.415671
2    0.471536
dtype: float64

In [4]:
from sklearn.metrics import average_precision_score
from tqdm import tqdm
import os


def score_portfolio(ensemble_algs):
    ap_scores = []
    for file, these_scores in scores.items():
        scores_array = np.concatenate([x.reshape(-1, 1) for n, x in these_scores.items() if n in ensemble_algs], axis=1)
        df = files[file]
        ap_scores.append(average_precision_score(df.Label, StandardScaler().fit_transform(scores_array).mean(axis=1)))
    return np.mean(ap_scores)

In [ ]:
selected_portfolio = []
best_so_far = 0
for i in range(7):
    print(best_so_far)
    new_scores = {}
    for new_alg in portfolio_big:
        if new_alg in selected_portfolio:
            continue
        new_scores[new_alg] = score_portfolio(selected_portfolio + [new_alg])
    print(new_scores)
    best_alg = pd.Series(new_scores).idxmax()
    print(f"best alg: {best_alg}")
    if new_scores[best_alg] > best_so_far:
        selected_portfolio.append(best_alg)
    else:
        break

In [59]:
selected_portfolio = ['Sub_KMeansAD',
 'SAND',
 'median_dist_score',
 'MatrixProfile',
 'Sub_PCA',
 'Decompose',
 'AutoEncoder',
 'median_dist_rolling_score'
  'Sub_LOF',
  'diff_diff_std_score']

In [45]:
selected_portfolio

['Sub_KMeansAD',
 'median_dist_score',
 'diff_diff_std_score',
 'median_dist_rolling_score',
 'median_dist_score',
 'median_dist_score']

In [46]:
small_portfolio = ['Sub_KMeansAD', 'median_dist_score', 'diff_diff_std_score', 'median_dist_rolling_score']
ap_scores_small_portfolio = {}
for file, these_scores in scores.items():
    scores_small = np.concatenate([x.reshape(-1, 1) for n, x in these_scores.items() if n in small_portfolio], axis=1)
    df = files[file]
    ap_scores_small_portfolio[file] = (average_precision_score(df.Label, StandardScaler().fit_transform(scores_small).mean(axis=1)))

In [47]:
pd.Series(ap_scores_small_portfolio).to_csv("ap_small_portfolio.csv")

In [9]:
selected_portfolio = []
best_so_far = 0
best_scores = []
for i in range(12):
    print(best_so_far)
    new_scores = {}
    for new_alg in portfolio_big:
        if new_alg in selected_portfolio:
            continue
        new_scores[new_alg] = score_portfolio(selected_portfolio + [new_alg])
    print(pd.Series(new_scores).sort_values())
    best_alg = pd.Series(new_scores).idxmax()
    print(f"best alg: {best_alg}")
    if new_scores[best_alg] > best_so_far:
        selected_portfolio.append(best_alg)
        best_so_far = new_scores[best_alg]
        best_scores.append(best_so_far)
    else:
        break

0
Sub_PCA_projection_quo_vadis     0.069996
Sub_PCA_projection               0.103504
negative_rolling_window_score    0.135225
negative_raw_signal              0.144587
KNN                              0.197041
rolling_diff_mult_1_5            0.222933
Sub_LOF                          0.232391
PCA                              0.234771
rolling_window_score             0.244037
AutoEncoder                      0.245650
rolling_3                        0.249014
Sub_PCA_projection2              0.251551
diff_rolling_score               0.257314
raw_signal                       0.262166
HBOS                             0.262577
MatrixProfile                    0.262642
POLY                             0.277365
diff_diff_std_score              0.279428
ucr_custom_average               0.283867
kpi_custom_average               0.283965
Sub_PCA_projection_quo_vadis2    0.287421
SR_with_window_size              0.298439
Sub_HBOS                         0.301790
Sub_MCD                         